# Ispravljanje grešaka u srpskom tekstu 

Pregled sva tri pristupa ispravljanja grešaka u kucanju:
1. **Rečnik + Levenštajnovo rastojanje** 
2. **N-gram rerangiranje** 
3. **LLM zero-shot** 

Prvi pristup zasniva se na frekvencijskom rečniku izgrađenom iz trening skupa. Reč koja se ne nalazi u rečniku smatra se potencijalnom greškom, nakon čega se kandidati za ispravke generišu u nekoliko nivoa: prvo se proveravaju kombinacije obnavljanja dijakritika (npr. c=>č/ć, s=>š, z=>ž, dj=>đ) , a zatim se, ukoliko rezultat i dalje nije poznata reč, generišu kandidati na Levenštajnovom rastojanju 1, pa tek potom 2. Od svih poznatih kandidata bira se onaj sa najvišom učestalošću u trening skupu, po ugledu na klasični Norvigov algoritam za proveru pravopisa.

Drugi pristup koristi potpuno isti postupak detekcije i generisanja
kandidata kao prvi, ali menja način na koji se među kandidatima bira
konačna ispravka. Umesto da se oslanja isključivo na učestalost reči van
konteksta, svaki kandidat se boduje bigramskim modelom koji uzima u obzir
prethodnu i narednu reč u rečenici, uz tehniku stupid backoff za slučajeve
kada određeni bigram nije viđen u trening skupu. Na taj način se
razrešavaju situacije u kojima je više kandidata podjednako verovatno kao
izolovana reč, ali samo jedan od njih zaista odgovara kontekstu rečenice.

Treći pristup se suštinski razlikuje od prva dva jer ne uključuje poseban
korak detekcije grešaka niti generisanje kandidata na nivou pojedinačnih
reči. Umesto toga, cela oštećena rečenica se prosleđuje velikom jezičkom
modelu u okviru jasno definisanog zero-shot prompta, koji modelu nalaže da
ispravi isključivo pravopisne i tipografske greške, bez izmene stila, reči
ili reda reči. Model zatim vraća ispravljenu rečenicu u celini, iz koje se
izmene naknadno poravnavaju sa originalnim tekstom kako bi se utvrdilo
tačno koja reč je i kako ispravljena.

Nakon pripreme trening skupa i generisanja sintetičkog test skupa sa kontrolisano
unetim greškama, svaki od pristupa primenjen je nad istim skupom rečenica, a
dobijeni rezultati statistički su upoređeni radi utvrđivanja koji pristup
postiže najbolju tačnost detekcije i ispravke grešaka po pojedinačnim tipovima
grešaka.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from common import  tokenize_with_spans, cyrillic_to_latin
from corrector_lib import load_freq_dict, correct_word, get_candidates, is_known, diacritic_candidates
from ngram_lib import BigramScorer, load_bigram_model, get_context
from errors_lib import ERROR_TYPES, corrupt_word, can_apply

pd.set_option("display.max_colwidth", 100)

## Početni podaci (pre obrade)

Pre bilo kakve obrade, izvorni podaci dolaze u vidu arhive koju
`scripts/01_prepare_corpus.py` preuzima sa Leipzig Wortschatz servera i čuva
lokalno kao `data/raw/srp_wikipedia_2021_1M.tar.gz`. Iz arhive se izdvaja
jedan tekstualni fajl (`srp_wikipedia_2021_1M-sentences.txt`) u kome je svaki
red oblika `<redni_broj><TAB><rečenica>` - bez ikakvog čišćenja: rečenice su
mešavina ćirilice i latinice, u originalnom obliku sa Wikipedije, ponegde sa
ostacima markupa.

Taj sirovi fajl se nalazi u `data/raw/` i predstavlja polaznu tačku pre
čišćenja, transliteracije, tokenizacije i uklanjanja duplikata opisanih u
Koraku 1 - dok se skripta ne pokrene, ni ova arhiva ni bilo koji od
obrađenih fajlova ne postoje na disku.

Ovo preuzimanje je potrebno isključivo za **treniranje**: iz njega se grade
frekvencijski rečnik i bigramski model na kojima se zasnivaju Pristup 1 i
Pristup 2. Podaci za **testiranje** (`data/test/`) su odvojeni i već postoje
u repozitorijumu - svi pristupi se na kraju ocenjuju nad istim, unapred
pripremljenim test skupom od 2.000 rečenica, opisanim u Koraku 2.

## Korak 1 - Trening skup

Trening skup: rečenice očišćene i prebačene sa ćirilice na latinicu
(transliterisane).

Rečenice se nalaze u fajlu `data/processed/train_sentences.txt` - običnom
tekstualnom fajlu u UTF-8 kodiranju, gde je svaka rečenica u posebnom redu,
bez ikakvih dodatnih oznaka ili metapodataka. Ovaj fajl se ne nalazi u
repozitorijumu - generiše ga skripta opisana ispod, i postoji tek nakon što
se ona pokrene.

Skup priprema skripta `scripts/01_prepare_corpus.py`, i to nad sirovim
fajlom opisanim u prethodnoj ćeliji: čisti tekst, transliteruje
ćirilicu na latinicu, tokenizuje rečenice i uklanja duplikate. Iz očišćenog
skupa izdvaja se i oko 2.000 rečenica koje se nikada ne koriste za izgradnju
rečnika ili n-gram modela, već se čuvaju posebno kao
`data/test/holdout_sentences.txt`. Taj fajl još uvek sadrži ispravne,
neoštećene rečenice . 
Tek u narednom koraku se u njih veštački ubacuju
greške, čime nastaje `data/test/test_set.jsonl`, konačni test skup nad kojim
se ocenjuju sva tri pristupa.

In [2]:
print((ROOT / "data" / "processed" / "corpus_stats.txt").read_text())

with open(ROOT / "data" / "processed" / "train_sentences.txt", encoding="utf-8") as f:
    sample_sentences = [next(f) for _ in range(5)]
for s in sample_sentences:
    print("-", s.strip())

Total cleaned sentences: 947312
Training sentences: 945312
Held-out sentences: 2000
Training tokens (incl. punctuation): 16898421
Training word tokens: 14312654

- 100 km severno od polarnog kruga.
- "100 najvećih gitarista svih vremena“, Hendriks se našao na prvom mestu.
- 100-serija se sastojala od proizvoda 9-serije sa drugačijim imenom.
- 10 – Odbaci paket i pošalji ICMP poruku o problemu sa parametima peketovoj izvorišnoj adresi, pokazujući na ne prepoznavanje tipa opcija.
- 11 – Odbaci paket i samo ako odredišna adresa nije multicast adresa, pošalji ICMP poruku o problemu sa parametrima izvorišnoj adresi paketa, pokazujući na neprepoznatljiv tip opcija.


Očišćeno ovde znači samo uklonjeno
formatiranje/markup i duplikati, transliterisana ćirilica i filtrirane
prekratke/predugačke rečenice - `clean_sentences()` u
`01_prepare_corpus.py` ne proverava pravopis. Trening skup zato može sadržati
i prave tipfelere iz izvornog Wikipedia teksta (npr. "peketovoj" umesto
"paketovoj") i ostatke markupa poput nekonzistentnih navodnika. Ako se takva
greška u izvoru ponovi dovoljno puta, može ući u frekvencijski rečnik kao
"poznata" reč.

### Transliteracija ćirilica => latinica

U srpskoj ćirilici tri slova - Љ, Њ i Џ - nemaju svoje pojedinačno latinično
slovo, već se transliterišu u po dva latinična karaktera: Љ=>Lj, Њ=>Nj, Џ=>Dž
(odnosno lj, nj, dž malim slovima). Ovakvi parovi karaktera zovu se digrafi.
Funkcija `cyrillic_to_latin()` (u `common.py`) mora prvo da prepozna i
zameni baš ova tri slova, pre nego što pređe na ostala, jednostavnim
mapiranjem slovo-za-slovo (npr. Ш=>Š) - u suprotnom bi digrafi mogli pogrešno
da se obrade kao dva odvojena, nepovezana slova.

Ćelija ispod testira funkciju na jednoj običnoj rečenici i na primeru koji
sadrži sva tri digrafa, kako bi se proverilo da li se ispravno prevode.

In [3]:
examples = ["Ово је реченица на ћирилици.", "Љубав, њега, џак - дигрèafи."]
for e in examples:
    print(e, "->", cyrillic_to_latin(e))

Ово је реченица на ћирилици. -> Ovo je rečenica na ćirilici.
Љубав, њега, џак - дигрèafи. -> Ljubav, njega, džak - digrèafi.


## Korak 2 - Sintetički test skup

Test skup generiše skripta `scripts/02_generate_testset.py`, nad rečenicama
izdvojenim u prethodnom koraku - onima koje nikada nisu korišćene pri
treniranju. U svaku rečenicu se ubacuje tačno jedna greška, jednog od pet
tipova: brisanje dijakritika, zamena susednim tasterom, brisanje karaktera,
umetanje karaktera ili transpozicija dva susedna karaktera.

Tip greške za svaku rečenicu bira pohlepni algoritam balansiranja: u svakom
trenutku se primenjuje onaj tip koji je trenutno najviše zaostao za svojim
ciljnim udelom od 20%. Na taj način broj grešaka po tipu ostaje izjednačen
tokom čitavog generisanja, bez potrebe da se unapred zna koliko će ukupno
rečenica biti obrađeno. Ukupno nastaje 2.000 rečenica, sa po 400 grešaka
svakog tipa.

In [4]:
with open(ROOT / "data" / "test" / "test_set.jsonl", encoding="utf-8") as f:
    test_set = [json.loads(line) for line in f]

print((ROOT / "data" / "test" / "test_set_stats.txt").read_text())

for rec in test_set[:6]:
    err = rec["errors"][0]
    print(f"[{err['type']:14s}] {err['bad_word']!r:20s} -> {err['orig_word']!r:20s} | {rec['corrupted']}")

Sentences generated: 2000
Sentences with >=1 injected error: 2000
Sentences skipped (no eligible word >= 3 chars): 0
Total injected errors: 2000
Average errors/sentence: 1.000
Error type distribution:
  diacritic        400  (20.0%)
  substitution     400  (20.0%)
  deletion         400  (20.0%)
  insertion        400  (20.0%)
  transposition    400  (20.0%)

[transposition ] 'lii'                -> 'ili'                | Naime, kao metamorfmafus, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak lii čarolije.
[substitution  ] 'Ssra'               -> 'Sara'               | Ssra ga je čekala i dobro ga izgrdila.
[deletion      ] 'tavih'              -> 'takvih'             | Institut takođe upravlja centrom za protonsku terapiju u Orseju, jednom od retkih tavih objekata na svetu.
[insertion     ] 'uspesih'            -> 'uspesi'             | Najznačajniji uspesih su mu ulazak u Super ligu Srbije 2002.
[diacritic     ] 'obicnog'            -> 'običnog'

Statistika iz `test_set_stats.txt` potvrđuje ravnomernu raspodelu opisanu u
prethodnom koraku: tačno 400 grešaka po svakom od pet tipova (ukupno 2.000,
jedna po rečenici, nijedna rečenica nije preskočena zbog nedostatka podobne
reči).

Šest ispisanih primera pokazuje kako svaki tip greške konkretno izgleda u
kontekstu cele rečenice: transpozicija (`lii` ↔ `ili`) i supstitucija
(`Ssra` ↔ `Sara`, zamena susednim tasterom) menjaju po jedan ili dva
karaktera unutar reči bez promene njene dužine, brisanje (`tavih` ← `takvih`)
i umetanje (`uspesih` ← `uspesi`, `sastavb` ← `sastav`) menjaju dužinu reči,
dok primer za dijakritike (`obicnog` ← `običnog`) pokazuje tipičan slučaj
pisanja bez č/ć/š/ž/đ.

### Generisanje grešaka

`errors_lib.py` sadrži pet funkcija za generisanje grešaka (po jednu za
svaki tip iz prethodnog koraka) i mapu susednih tastera koju koriste
supstitucija i umetanje. Mapa se gradi iz tri reda srpske Latinice na
QWERTZ tastaturi (`qwertzuiopšđ`, `asdfghjklčć`, `yxcvbnm`), tako da svako
slovo dobije skup suseda: levi i desni sused u istom redu, plus
proporcionalno poravnati susedi u redu iznad/ispod (redovi nisu iste
dužine). Zahvaljujući tome, kad se npr. slovo "s" pokvari supstitucijom,
kandidat je stvarno susedno slovo ("a", "d", "š"...), a ne nasumično slovo
iz cele azbuke - što realističnije oponaša prave tipfelere.

Ćelija ispod primenjuje svih pet tipova greške na istu reč (`šetnja`) i
ispisuje rezultat svakog: dijakritika uklanja kvačicu (`šetnja=>setnja`),
supstitucija menja jedno slovo susednim (`šetnja=>ćetnja`), brisanje
uklanja jedan karakter (`šetnja=>šetna`), umetanje dodaje karakter susedan
nekom postojećem (`šetnja=>šetnmja`), a transpozicija zamenjuje mesta dva
susedna karaktera (`šetnja=>šetjna`).

In [5]:
import random
rng = random.Random(0)
word = "šetnja"
for t in ERROR_TYPES:
    if can_apply(word, t):
        print(f"{t:14s}: {word} -> {corrupt_word(word, t, rng)}")

diacritic     : šetnja -> setnja
substitution  : šetnja -> ćetnja
deletion      : šetnja -> šetna
insertion     : šetnja -> šetnmja
transposition : šetnja -> šetjna


## Korak 3 - Pristup 1: Rečnik + Levenštajnovo rastojanje 

Detekcija greške je jednostavna: reč se smatra greškom ako se ne nalazi u
frekvencijskom rečniku izgrađenom iz trening skupa.

Ako reč nije poznata, kandidati za ispravku generišu se u tri nivoa, po
redu, a pretraga se zaustavlja čim se na nekom nivou pronađe bar jedan
poznat kandidat:

1. **obnavljanje dijakritika** - reč se tretira kao da su joj kvačice
   izbrisane, pa se probaju sve kombinacije vraćanja č/ć, š, ž i đ;
2. **rastojanje uređivanja (Levenštajn) 1** - sve reči koje se od unete
   razlikuju za tačno jedno brisanje, umetanje, zamenu ili transpoziciju
   karaktera;
3. **rastojanje uređivanja 2** - isto, ali dozvoljene su dve takve izmene.

Od svih poznatih kandidata pronađenih na tom (najranijem) nivou bira se onaj
sa najvišom učestalošću u trening skupu.

Pristup implementira skripta `scripts/03_norvig_corrector.py`, uz pomoćne
funkcije iz `corrector_lib.py` (izgradnja frekvencijskog rečnika i
generisanje kandidata na sva tri nivoa). Kako je detekcija zasnovana
isključivo na tome da li je reč poznata, pristup ne može da otkrije grešku
koja slučajno formira drugu, već postojeću reč - primer u nastavku pokazuje
baš takav slučaj (`medju`, umesto `među`, prepoznato je kao poznata reč i
ostaje neizmenjeno).

In [6]:
freq = load_freq_dict(ROOT / "data" / "processed" / "word_freq.json")
print(f"Vocabulary size: {len(freq)} word types")

Vocabulary size: 588767 word types


### Namenski generator za obnavljanje dijakritika

Kada neko piše bez kvačica, gubitak dijakritike nije nasumična greška u
tipkanju - svako slovo `c`, `s`, `z` i par `dj` ima tačno određen, mali skup
mogućnosti odakle je moglo nastati: `c` je moglo biti `c`, `č` ili `ć`; `s`
je moglo biti `s` ili `š`; `z` je moglo biti `z` ili `ž`; a `dj` je moglo
biti `dj` ili `đ`. Funkcija `diacritic_candidates()` prolazi kroz reč
slovo po slovo, za svako od ovih mesta izlistava mogućnosti, a zatim
generiše sve kombinacije tih izbora (Dekartov proizvod) - ostala slova
ostaju nepromenjena.

Zašto ovo postoji kao poseban korak, a ne samo kao deo pretrage po
rastojanju uređivanja: ako je u jednoj reči izgubljeno više dijakritika
odjednom (npr. `razlicit` umesto `različit` - nedostaju dve kvačice),
ispravna reč je već na rastojanju uređivanja 2 od unete, na samoj granici do
koje se inače traži. Kod reči sa tri ili više izgubljenih dijakritika,
ispravna reč bi bila van tog dometa i generički pretraživač je ne bi
pronašao. Namenski generator to rešava direktno - ne mora da pretražuje sve
moguće izmene reči, već samo kombinuje unapred poznate zamene na tačno
određenim mestima, pa je i brz i tačan bez obzira koliko je dijakritika
nestalo.

In [7]:
for w in ["necu", "svidja", "razlicit"]:
    cands = diacritic_candidates(w)
    known_cands = {c: freq.get(c, 0) for c in cands if freq.get(c, 0) > 0}
    print(f"{w:12s} -> {len(cands)} candidates, known ones: {known_cands}")

necu         -> 2 candidates, known ones: {'neću': 77}
svidja       -> 3 candidates, known ones: {'sviđa': 71}
razlicit     -> 5 candidates, known ones: {'različit': 210}


In [8]:
examples = ["obicnog", "vecinu", "medju", "tavih", "napazi", "koledz"]
for w in examples:
    known = is_known(w, freq)
    corrected, changed = correct_word(w, freq)
    print(f"{w:12s} known={known!s:6s} -> corrected: {corrected!r} (changed={changed})")

obicnog      known=False  -> corrected: 'običnog' (changed=True)
vecinu       known=False  -> corrected: 'većinu' (changed=True)
medju        known=True   -> corrected: 'medju' (changed=False)
tavih        known=False  -> corrected: 'takvih' (changed=True)
napazi       known=False  -> corrected: 'nalazi' (changed=True)
koledz       known=False  -> corrected: 'koledž' (changed=True)


Rečnik izgrađen iz trening skupa ima skoro 589.000 poznatih reči (uključujući
sve padežne oblike, pošto se ne radi lematizacija). Od šest primera, pet je
ispravljeno: `obicnog`, `vecinu` i `koledz` rešeni su već na nivou obnavljanja
dijakritika (c=>ć, z=>ž), dok su `tavih` i `napazi` rešeni na rastojanju
uređivanja 1 (jedno brisanje, odnosno jedna zamena karaktera).

Jedini izuzetak je `medju` - tačno onaj slučaj najavljen u prethodnoj ćeliji:
reč postoji u rečniku (verovatno kao alternativni/nestandardni zapis reči
"među" koji se dovoljno puta pojavio u trening skupu), pa je `known=True` i
korektor je uopšte ne dira, iako je u pitanju greška bez dijakritike.

## Korak 4 - Pristup 2: N-gram rerangiranje

Detekcija greške i generisanje kandidata identični su Pristupu 1 (isti
frekvencijski rečnik, ista tri nivoa kandidata). Razlika je isključivo u
tome kako se, među već pronađenim poznatim kandidatima, bira konačna
ispravka.

Umesto da se, kao u Pristupu 1, uvek uzme kandidat sa najvišom opštom
učestalošću u korpusu, ovde se svaki kandidat boduje u odnosu na susedne
reči u samoj rečenici: bod = log P(kandidat | prethodna reč) + log
P(naredna reč | kandidat). Te verovatnoće računaju se iz bigramskog modela
izgrađenog nad trening skupom - brojanjem koliko puta se koja reč pojavila
(unigram) i koliko puta se koja reč pojavila neposredno posle koje druge
(bigram). Kad specifičan par susednih reči nikad nije viđen u trening
skupu, model ne pada na nulu, već koristi tehniku stupid backoff: tu
verovatnoću aproksimira sa 0.4 puta.

Reč na početku ili kraju rečenice nema jednog suseda (prethodnu, odnosno
narednu reč) - zato se rečenica interno uokviruje posebnim oznakama `<s>`
(početak) i `</s>` (kraj), koje se u modelu ponašaju kao "susedne reči",
tako da i prva i poslednja reč u rečenici i dalje dobijaju smislen (makar
jednostran) bigramski kontekst, umesto da budu izostavljene iz bodovanja.

Pristup implementira skripta `scripts/04_ngram_corrector.py`, uz bigramski
model iz `ngram_lib.py`.

In [9]:
unigrams, bigrams = load_bigram_model(ROOT / "data" / "processed" / "bigram_model.json")
scorer = BigramScorer(unigrams, bigrams)

sentence = "Nakon rata se u seno vratilo manje od stotinu ljudi."
tws = tokenize_with_spans(sentence)
idx = next(i for i, (t, _, _) in enumerate(tws) if t == "seno")
prev_word, next_word = get_context(tws, idx)
print("context:", prev_word, "[?]", next_word)

candidates = get_candidates("seno", freq)
print("known candidates:", candidates)
for c in candidates:
    print(f"  {c:10s} unigram_freq={freq.get(c,0):6d}  bigram_score={scorer.score(c, prev_word, next_word):.3f}")

context: u [?] vratilo
known candidates: {'seno'}
  seno       unigram_freq=    39  bigram_score=-25.858


 Reč "seno" je sama po sebi poznata reč u
rečniku, pa `get_candidates()` odmah vraća `{'seno'}` kao
jedini kandidat - prvi nivo pretrage u `candidate_tiers()` proverava baš to,
da li je uneta reč već poznata, i ako jeste, pretraga se tu zaustavlja.
Nikad se ne stiže do konkurentskih kandidata poput "selo", koju rečenica
verovatno traži ("vratilo se u selo", ne "u seno").

Ovo je primer real-word greške: reč je pogrešna u kontekstu, ali je i dalje
validna srpska reč, pa je nijedan od ova dva pristupa ne prepoznaje kao
grešku - detekcija se zasniva isključivo na tome da li je reč poznata
rečniku, a ne na tome da li ima smisla u konkretnoj rečenici. Da bi se
pokazalo pravo razrešavanje dvosmislenosti (unigram vs. bigram pobednik),
bio bi potreban primer sa rečju koja nije poznata i za koju postoji više
poznatih kandidata na rastojanju uređivanja.

## Korak 5 - Pristup 3: LLM zero-shot

Ovaj pristup se suštinski razlikuje od prva dva: ne postoji poseban korak
detekcije niti generisanje kandidata za pojedinačne reči. Umesto toga, cela
oštećena rečenica se šalje velikom jezičkom modelu uz fiksni prompt koji
traži da model ispravi isključivo pravopisne i tipografske greške, bez
menjanja stila, reči ili reda reči, i da odgovori samo ispravljenom
rečenicom, bez uvoda ili objašnjenja - prompt je prikazan u ćeliji ispod
ove sekcije.

Pristup implementira skripta `scripts/05_llm_corrector.py`, uz `llm_lib.py`,
koji nudi tri međusobno zamenljiva provajdera (Anthropic, OpenAI, Groq) preko
istog jednostavnog interfejsa (`LLMClient.complete()`) - izbor modela se
svodi na promenu jednog argumenta pri pokretanju skripte. Rezultati
prikazani u ovoj svesci dobijeni su preko Groq-a, modelom
`llama-3.3-70b-versatile`.

Kako svaki poziv modelu košta vreme i, kod pojedinih provajdera, novac,
svaki odgovor se kešira na disk pod ključem izvedenim iz same rečenice
(`sentence_key()`), u fajlu `data/test/llm_cache/<provajder>_<model>.json`.
Ponovljeno pokretanje skripte tako nikad ne plaća isti upit dvaput, već
samo dopunjuje keš onim što još nije obrađeno - ćelija ispod čita taj keš
direktno sa diska, pa prikazuje primere čak i ako obrada u pozadini još
nije završena.

Pošto model vraća slobodan tekst cele rečenice, a ne listu izmena po
rečima, `align_changes()` naknadno poravnava taj odgovor sa originalnim
tokenima rečenice pomoću `difflib`, kako bi se rekonstruisalo koje su reči
i na kojoj poziciji zapravo izmenjene - to je neophodno da bi se ovaj
pristup uopšte mogao meriti istom metrikom (tačnost po reči) kao Pristup 1
i 2.

In [ ]:
from llm_lib import sentence_key, clean_response, build_prompt

cache_files = list((ROOT / "data" / "test" / "llm_cache").glob("*.json"))
cache_files = [p for p in cache_files if "progress" not in p.name]
print("cache files found:", [p.name for p in cache_files])

if cache_files:
    cache = json.loads(cache_files[0].read_text(encoding="utf-8"))
    print(f"{len(cache)} sentences cached so far")

    shown = 0
    for rec in test_set:
        k = sentence_key(rec["corrupted"])
        if k in cache and shown < 5:
            raw = cache[k]
            predicted = clean_response(raw)
            err = rec["errors"][0]
            print(f"corrupted: {rec['corrupted']}")
            print(f"LLM said : {predicted}")
            print(f"original : {rec['original']}")
            print(f"error: {err['bad_word']} -> {err['orig_word']} ({err['type']})")
            print()
            shown += 1
else:
    print("No LLM cache yet - run scripts/05_llm_corrector.py first.")

cache files found: ['groq_llama-3.3-70b-versatile_llm.json']
2000 sentences cached so far
corrupted: Naime, kao metamorfmafus, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak lii čarolije.
LLM said : Naime, kao metamorfmafs, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak ili čarolije.
original : Naime, kao metamorfmafus, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak ili čarolije.
error: lii -> ili (transposition)

corrupted: Ssra ga je čekala i dobro ga izgrdila.
LLM said : Sestra ga je čekala i dobro ga izgrdila.
original : Sara ga je čekala i dobro ga izgrdila.
error: Ssra -> Sara (substitution)

corrupted: Institut takođe upravlja centrom za protonsku terapiju u Orseju, jednom od retkih tavih objekata na svetu.
LLM said : Institut takođe upravlja centrom za protonsku terapiju u Orseju, jednom od retkih takvih objekata na svetu.
original : Institut takođe upravlja centrom

/Users/natalijapavlovic/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Od pet prikazanih primera, tri su ispravljena tačno onako kako je i
očekivano: `tavih=>takvih`, `uspesih` je barem prepoznat kao problem, i `obicnog=>običnog` se poklapa reč za reč sa
originalom. Ostala dva pokazuju konkretne slabosti ovog pristupa:

- **`Ssra => Sara`**: model je vratio `Sestra` umesto `Sara`. Gramatički i
  semantički rečenica i dalje ima smisla ("Sestra ga je čekala..."), ali se
  ne poklapa sa stvarno ubačenom greškom - model je pogodio "verovatnu"
  reč, a ne nužno onu koja je originalno stajala tu.
- **`uspesih => uspesi`**: model je vratio `uspeh` (jednina) umesto `uspesi`
  (množina), iako ostatak rečenice ("... su mu ulazak...") gramatički traži
  množinu. I dalje je zvučalo dovoljno prirodno da ga model nije ispravio
  na traženi oblik.
- **Prva rečenica sadrži i neželjenu izmenu**: uz traženi ispravak
  (`lii=>ili`), model je promenio i `metamorfmafus` u `metamorfmafs` - reč
  koja uopšte nije bila označena kao greška. Ovo je konkretan primer
  prekomerne ispravke (overcorrection) pomenute u uvodu: model ponekad menja
  i ono što nije trebalo da dira.

### Primer prompta poslatog modelu

In [11]:
print(build_prompt(test_set[0]["corrupted"]))

Ispravi isključivo pravopisne i tipografske greške u sledećoj rečenici (dijakritike, tipfelere). Ne menjaj stil, reči ni red reči osim da ispraviš greške. Odgovori isključivo ispravljenom rečenicom - bez uvoda, objašnjenja, navodnika ili napomena.

Rečenica: Naime, kao metamorfmafus, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak lii čarolije.
Ispravljena rečenica:


## Korak 6 - Evaluacija

Evaluaciju sprovodi skripta `scripts/06_evaluate.py`, koja za svaki pristup
za koji postoji fajl sa predikcijama (`results/predictions/<pristup>.jsonl`) računa sledeće mere, na osnovu poznatih injektovanih
grešaka iz test skupa:

- **detekcioni recall** - od svih injektovanih grešaka, koliki procenat je
  pristup uopšte primetio i nešto promenio na tom mestu (bez obzira da li je
  ispravka tačna);
- **tačnost ispravke** (glavna mera) - od svih injektovanih grešaka, koliki
  procenat je ispravljen na tačno onu reč koja je originalno stajala tu;
- **tačnost po tipu greške** - ista mera, posebno za svaki od pet tipova
  greške (dijakritika, supstitucija, brisanje, umetanje, transpozicija);
- **stopa lažno pozitivnih ispravki** - od svih reči koje *nisu* bile
  greška, koliki procenat je pristup ipak izmenio (prekomerna ispravka);
- **brzina obrade** (rečenica/sek) - iz `_stats.txt` fajla koji je svaka
  skripta upisala tokom sopstvenog izvršavanja.

In [12]:
results_path = ROOT / "results" / "results.csv"
if results_path.exists():
    df = pd.read_csv(results_path)
    display(df)
else:
    print("results.csv not found - run scripts/06_evaluate.py first.")

,approach,detection_recall,correction_accuracy,false_positive_rate,sentences_per_sec,acc_diacritic,acc_substitution,acc_deletion,acc_insertion,acc_transposition
0,Dictionary + Edit Distance,0.8005,0.689,0.018669,94.98,0.6675,0.6975,0.3925,0.9125,0.775
1,N-gram Reranking,0.8005,0.728,0.018669,94.65,0.6700,0.7600,0.4675,0.9175,0.825
2,LLM Zero-shot,0.9415,0.781,0.024273,2.52,0.8300,0.7000,0.7500,0.8300,0.795


**Ngram protiv Rečnika**: identičan detekcioni recall (0.800) i identična
stopa lažno pozitivnih (0.019) potvrđuju da je detekcija zaista ista kod
oba pristupa. Tačnost ispravke ipak
raste sa 0.689 na 0.728, i to dosledno na svih pet tipova grešaka, najviše
kod brisanja (0.393=>0.468): to je čist efekat bigramskog konteksta pri
biranju između kandidata.

**LLM protiv oba lokalna pristupa**: ima ubedljivo najveći detekcioni
recall (0.942 naspram 0.800) i najveću tačnost ispravke (0.781) -
jer ne zavisi od toga da li je reč "poznata" (baš zato hvata i real-word
greške tipa `seno`/`selo` iz primera iznad). Cena je dvostruka: nešto viša
stopa lažno pozitivnih (0.024 naspram 0.019, tj. povremeno menja i reči
koje nisu bile greška), i
drastično manja brzina (2.52 rečenice/sek naspram ~95 kod druga dva
pristupa, dakle oko 38 puta sporije, jer svaka rečenica ide preko mrežnog
poziva ka API-ju).

LLM ubedljivo dominira baš tamo gde su Rečnik i Ngram najslabiji - dijakritika (0.830 naspram ~0.67) i
posebno brisanje (0.750 naspram 0.39–0.47), gde tačna reč nije uvek
najbliža po rastojanju uređivanja. Ali kod umetanja (0.830 naspram 0.917
kod Ngrama) i transpozicije (0.795 naspram 0.825 kod Ngrama), LLM je
zapravo **lošiji** od oba lokalna pristupa - u tim slučajevima je oštećena
reč toliko blizu tačnoj (jedan karakter viška ili zamenjena dva mesta) da
je rečnički pristup gotovo nepogrešiv, dok LLM ume da "ispravi" rečenicu na
drugačiji, i dalje ispravan, ali ne i identičan način.

### Statistička značajnost

Tabela iznad pokazuje da je LLM tačniji od Ngrama, koji je tačniji od
Rečnika - ali da li je ta razlika stvarna, ili bi se mogla desiti i da su
pristupi podjednako dobri, samo zbog toga koje je konkretne greške test
skup sadržao? To proverava McNemar test.

Za svaki par pristupa, test posmatra samo greške na kojima se njih dvoje
*ne slažu* - gde je jedan pristup tačno ispravio grešku, a drugi nije.
Greške koje su oba pristupa ispravila tačno, ili oba pogrešila, se
ignorišu, jer ne govore ništa o tome koji je pristup bolji. Iz broja
slučajeva "samo A je tačan" i "samo B je tačan" (kolone `a_only_correct` i
`b_only_correct` u tabeli ispod), tačan binomni McNemar test računa
verovatnoću (p-vrednost) da bi tolika neravnoteža nastala slučajno, da su
pristupi u stvari podjednako dobri - što je ta verovatnoća manja od
uobičajenog praga 0.05, to je manje verovatno da je razlika slučajna.

Test je *uparen*: svaki pristup se poredi na potpuno istim greškama, iz
istog zajedničkog podskupa. To ga čini osetljivijim od
prostog poređenja dve tačnosti, jer koristi informaciju o tome koji su
konkretni slučajevi doprineli razlici, a ne samo agregatni procenat.

In [13]:
sig_path = ROOT / "results" / "significance.csv"
if sig_path.exists():
    display(pd.read_csv(sig_path))
else:
    print("significance.csv not found - run scripts/06_evaluate.py first.")

,approach_a,approach_b,a_only_correct,b_only_correct,p_value,significant_at_0.05
0,Dictionary + Edit Distance,N-gram Reranking,14,92,3.060238e-15,True
1,Dictionary + Edit Distance,LLM Zero-shot,249,433,1.792645e-12,True
2,N-gram Reranking,LLM Zero-shot,266,372,3.101031e-05,True


Sva tri poređenja izlaze statistički značajna (p ≪ 0.05), što znači da
poredak tačnosti iz prethodne tabele (LLM > Ngram > Rečnik) nije slučajnost
uzorkovanja - potvrđuje se i na nivou pojedinačnih grešaka, ne samo u
agregatnim procentima.

Vredi pogledati i same brojeve diskordantnih parova, ne samo p-vrednost:

- **Rečnik vs. Ngram**: 14 grešaka gde je samo Rečnik tačan, naspram 92 gde
  je samo Ngram tačan. Ngram gotovo nikad ne "pokvari" nešto što je Rečnik
  već dobro pogodio (svega 14 slučajeva) - bigramsko rerangiranje je,
  drugim rečima, skoro isključivo dobitak, retko kad šteta.
- **Rečnik vs. LLM**: 249 naspram 433 - LLM pobeđuje ubedljivo, ali i dalje
  postoji 249 grešaka koje je *samo* Rečnik tačno pogodio, a LLM promašio.
- **Ngram vs. LLM**: 266 naspram 372 - najmanja razlika u odnosu (razmera
  ~1.4:1, naspram ~1.7:1 i ~6.6:1 kod druga dva para), što se slaže sa
  ranijim nalazom da LLM kod umetanja i transpozicije zapravo zaostaje za
  Ngramom. Tih 266 grešaka gde je Ngram tačan, a LLM nije, uglavnom dolaze
  baš iz tih tipova grešaka.

Ukratko: LLM je u proseku najbolji, ali "u proseku" ovde krije stvarnu
komplementarnost - Rečnik i Ngram i dalje tačno pogađaju desetine grešaka
koje LLM promaši.

![Ukupna tačnost po pristupu](results/charts/overall_accuracy.png)

![Tačnost po tipu greške](results/charts/per_error_type.png)

Oba grafikona vizuelizuju brojeve iz `results.csv`. prvi (`overall_accuracy.png`) je prosta uporedna slika tri stuba
(0.69 / 0.73 / 0.78), dok drugi (`per_error_type.png`) grupiše po pet
tipova grešaka i tri pristupa jedan pored drugog. Na drugom grafikonu se
najbolje vidi obrazac koji je teško uočiti samo iz tabele brojeva: stubac
LLM-a je najviši kod dijakritike i brisanja, ali kod umetanja i
transpozicije ga vidno prestignu i Rečnik i Ngram - tačno ona
komplementarnost pristupa opisana u prethodnim ćelijama.

### Dodatni grafici

![Detekcioni recall vs. tačnost ispravke](results/charts/detection_vs_accuracy.png)

Rečnik i Ngram imaju identičan recall (0.800, po dizajnu - dele istu detekciju), ali se
razilaze u tačnosti, LLM ima znatno veći recall (0.942) jer ne zavisi od toga
da li je reč "poznata" već procenjuje celu rečenicu.

![Stopa preterane ispravke](results/charts/false_positive_rate.png)

Cena tog većeg recall-a kod LLM-a: on i najviše menja reči koje uopšte nisu
bile greška (overcorrection) - vidi se i konkretno u `error_analysis.md`.

![Brzina po pristupu (logaritamska skala)](results/charts/speed_comparison.png)

Razlika u brzini je toliko velika (Rečnik/Ngram ~95 rečenica/sek naspram LLM
~2.52) da bi na linearnoj skali LLM stub bio nevidljiv - otuda logaritamska
skala.


### Uporedo: iste rečenice, tri modela

Do sada su brojevi tačnosti bili apstraktni procenti. Ova ćelija ih pretvara
u konkretne primere: za 10 pojedinačnih grešaka (dva primera po svakom od
pet tipova) prikazuje se, jedno pored drugog, šta je svaki od tri pristupa
(Rečnik, Ngram, LLM) predvideo na mestu te iste greške.

Primeri su uzeti iz zajedničkog podskupa rečenica koje su obradile sve tri
metode.

Kolone `norvig_ok`, `ngram_ok` i `llm_ok` govore da li se predviđanje
poklapa sa tačnom rečju - red po red, gde je koji
pristup pogodio, a gde promašio, umesto da se to zaključuje samo iz
agregatnih procenata.

In [14]:
from collections import defaultdict

def load_predictions(name):
    path = ROOT / "results" / "predictions" / f"{name}.jsonl"
    preds = [json.loads(line) for line in open(path, encoding="utf-8")]
    idx_path = ROOT / "results" / "predictions" / f"{name}_indices.json"
    indices = json.loads(idx_path.read_text(encoding="utf-8")) if idx_path.exists() else list(range(len(test_set)))
    return dict(zip(indices, preds))

def predicted_word(preds_by_index, i, idx):
    changes = {c["index"]: c["to"] for c in preds_by_index[i]["changes"]}
    return changes.get(idx, "(unchanged)")

norvig_preds = load_predictions("norvig")
ngram_preds = load_predictions("ngram")
llm_preds = load_predictions("llm")
common = set(norvig_preds) & set(ngram_preds) & set(llm_preds)

by_type = defaultdict(list)
for i in sorted(common):
    for err in test_set[i]["errors"]:
        by_type[err["type"]].append((i, err))

rows = []
for t in ERROR_TYPES:
    for i, err in by_type[t][:2]:
        idx, correct_word = err["index"], err["orig_word"]
        preds = {
            "norvig": predicted_word(norvig_preds, i, idx),
            "ngram": predicted_word(ngram_preds, i, idx),
            "llm": predicted_word(llm_preds, i, idx),
        }
        row = {"type": t, "bad_word": err["bad_word"], "correct_word": correct_word}
        for name, pred in preds.items():
            row[name] = pred
            row[f"{name}_ok"] = pred.lower() == correct_word.lower()
        rows.append(row)

comparison_df = pd.DataFrame(rows)
display(comparison_df)

,type,bad_word,correct_word,norvig,norvig_ok,ngram,ngram_ok,llm,llm_ok
0,diacritic,obicnog,običnog,običnog,True,običnog,True,običnog,True
1,diacritic,koledz,koledž,koledž,True,koledž,True,koledž,True
2,substitution,Ssra,Sara,Sara,True,Sara,True,Sestra,False
3,substitution,zyjednice,zajednice,zajednice,True,zajednice,True,zjednice,False
4,deletion,tavih,takvih,takvih,True,takvih,True,takvih,True
5,deletion,tv,tzv,(unchanged),False,(unchanged),False,(unchanged),False
6,insertion,uspesih,uspesi,uspesi,True,uspesi,True,uspeh,False
7,insertion,sastavb,sastav,sastavu,False,sastavu,False,sastav,True
8,transposition,lii,ili,(unchanged),False,(unchanged),False,ili,True
9,transposition,obji,boji,boji,True,boji,True,boji,True


Od 10 primera, obrazac se jasno deli u četiri grupe:

- **Sva tri tačna (4 od 10)**: redovi 0, 1, 4, 9 - `obicnog`, `koledz`,
  `tavih`, `obji`. Ovo su "lake" greške gde postoji tačno jedan poznat
  kandidat i sve tri metode ga nalaze.
- **Sva tri pogrešna (1 od 10)**: red 5, `tv`=>`tzv`. Nijedan pristup nije
  ni promenio reč. `tv` je sama po sebi kratka, česta reč
  (skraćenica za televiziju), pa je nijedan rečnički pristup ne prepoznaje
  kao grešku. Izgleda da je i LLM u ovoj instanci propustio da je ispravi.
- **Rečnik i Ngram tačni, LLM pogrešan (3 od 10)**: redovi 2, 3, 6 -
  `Ssra=>Sestra`, `zyjednice=>zjednice`, `uspesih=>uspeh`. Ovde je zanimljivo
  da LLM u redu 3 nije samo promašio tačnu reč, već je uneo i **novu**
  grešku (`zjednice` nije ni tačna reč `zajednice`, ni originalna
  `zyjednice`).
- **LLM tačan, Rečnik i Ngram pogrešni (2 od 10)**: redovi 7 i 8 -
  `sastavb=>sastav` i `lii=>ili`. Ovde su oba lokalna pristupa promašila na
  različite načine: kod `sastavb` su izabrala pogrešnog kandidata istog
  ranga, a kod `lii` uopšte
  nisu prepoznala/promenila reč - LLM je u oba slučaja pogodio tačno.

Ovih 10 slučajeva je premali uzorak za zaključke sam po sebi, ali se
uklapaju tačno u obrazac koji je već viđen u agregatnim brojevima: LLM
dominira ali nije nepogrešiv, a Rečnik/Ngram i dalje hvataju stvari koje
LLM promaši.

In [15]:
for t in ERROR_TYPES:
    for i, err in by_type[t][:2]:
        rec = test_set[i]
        print(f"[{t}]  {err['bad_word']!r} -> should be {err['orig_word']!r}")
        print(f"  corrupted : {rec['corrupted']}")
        print(f"  original  : {rec['original']}")
        print(f"  norvig    : {norvig_preds[i]['predicted']}")
        print(f"  ngram     : {ngram_preds[i]['predicted']}")
        print(f"  llm       : {llm_preds[i]['predicted']}")
        print()

[diacritic]  'obicnog' -> should be 'običnog'
  corrupted : Svi ga tretiraju kao obicnog zeca sem Maksa kome je Koloso dobar drug.
  original  : Svi ga tretiraju kao običnog zeca sem Maksa kome je Koloso dobar drug.
  norvig    : Svi ga tretiraju kao običnog zeca sem Maksa kome je Kolosa dobar drug.
  ngram     : Svi ga tretiraju kao običnog zeca sem Maksa kome je Kolos dobar drug.
  llm       : Svi ga tretiraju kao običnog zeca sem Maksa kome je Koloso dobar drug.

[diacritic]  'koledz' -> should be 'koledž'
  corrupted : Rezultat toga je Kraljevski koledz za negu (RKN), a RBNA je izgubila članstvo i dominaciju.
  original  : Rezultat toga je Kraljevski koledž za negu (RKN), a RBNA je izgubila članstvo i dominaciju.
  norvig    : Rezultat toga je Kraljevski koledž za negu (RNK), a RBNA je izgubila članstvo i dominaciju.
  ngram     : Rezultat toga je Kraljevski koledž za negu (RNK), a RBNA je izgubila članstvo i dominaciju.
  llm       : Rezultat toga je Kraljevski koledž za negu (RKN

## Korak 7 - Analiza grešaka

Skripta `scripts/07_error_analysis.py` iz istih predikcija koje ocenjuje
`06_evaluate.py` izdvaja do 10 konkretnih primera za svaku od četiri
kategorije, radi kvalitativnog uvida koji brojevi sami po sebi ne pokazuju:

- **(a) greške koje je ispravio samo LLM** - Rečnik i Ngram su oba
  pogrešila, LLM je pogodio;
- **(b) greške koje je Ngram ispravio, a Rečnik nije** - direktna
  demonstracija vrednosti bigramskog konteksta iz Koraka 4;
- **(c) greške koje nijedan dostupan pristup nije ispravio** - najteži
  slučajevi, gde je tačna reč van domašaja svih metoda;
- **(d) LLM-ove prekomerne ispravke** - slučajevi gde je LLM izmenio reč
  koja uopšte nije bila deo injektovane greške.

Analiza koristi zajednički podskup rečenica koje su
obradili svi dostupni pristupi.

In [16]:
analysis_path = ROOT / "results" / "error_analysis.md"
if analysis_path.exists():
    display(Markdown(analysis_path.read_text(encoding="utf-8")))
else:
    print("error_analysis.md not generated yet - run scripts/07_error_analysis.py first.")

# Error analysis

## (a) Cases only the LLM got right

**Example 1**

- corrupted: `Naime, kao metamorfmafus, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak lii čarolije.`
- original: `Naime, kao metamorfmafus, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak ili čarolije.`
- error: `lii` → should be `ili` (type: transposition)
- LLM predicted: `Naime, kao metamorfmafs, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak ili čarolije.`

**Example 2**

- corrupted: `Postojalo je od VI do X veka, kada je ušlo u sastavb jedinstvene Engleske države.`
- original: `Postojalo je od VI do X veka, kada je ušlo u sastav jedinstvene Engleske države.`
- error: `sastavb` → should be `sastav` (type: insertion)
- LLM predicted: `Postojalo je od VI do X veka, kada je ušlo u sastav jedinstvene Engleske države.`

**Example 3**

- corrupted: `Stalno savetuje Ivana da ne ini greške.`
- original: `Stalno savetuje Ivana da ne čini greške.`
- error: `ini` → should be `čini` (type: deletion)
- LLM predicted: `Stalno savetuje Ivana da ne čini greške.`

**Example 4**

- corrupted: `Molekul glukoze zasladjuje rastvor, a dijelovi glukoznih lanaca daju viskoznu konzistenciju.`
- original: `Molekul glukoze zaslađuje rastvor, a dijelovi glukoznih lanaca daju viskoznu konzistenciju.`
- error: `zasladjuje` → should be `zaslađuje` (type: diacritic)
- LLM predicted: `Molekul glukoze zaslađuje rastvor, a delovi glukoznih lanaca daju viskoznu konzistenciju.`

**Example 5**

- corrupted: `Anastasija je sledila svog sina, međutm uskoro se povukla iz javnog života i postala je opatica u Admont opatiji gde je ostala sve do svoje smrti.`
- original: `Anastasija je sledila svog sina, međutim uskoro se povukla iz javnog života i postala je opatica u Admont opatiji gde je ostala sve do svoje smrti.`
- error: `međutm` → should be `međutim` (type: deletion)
- LLM predicted: `Anastasija je sledila svog sina, međutim uskoro se povukla iz javnog života i postala je opatica u Admont opatiji, gde je ostala sve do svoje smrti.`

**Example 6**

- corrupted: `Nakon rata se u seno vratilo manje od stotinu ljudi, uglavnom starijih osoba.`
- original: `Nakon rata se u selo vratilo manje od stotinu ljudi, uglavnom starijih osoba.`
- error: `seno` → should be `selo` (type: substitution)
- LLM predicted: `Nakon rata se u selo vratilo manje od stotinu ljudi, uglavnom starijih osoba.`

**Example 7**

- corrupted: `Marina Nikola je medju prvima 1927.`
- original: `Marina Nikola je među prvima 1927.`
- error: `medju` → should be `među` (type: diacritic)
- LLM predicted: `Marina Nikola je među prvimama 1927.`

**Example 8**

- corrupted: `Tipične analize su analize prolaznih podataka za pravljenje upotrebno-definisanih lanaca, zavisnih analiza, alijas analiza, pokaznih analiza, iskejp analiza ird.`
- original: `Tipične analize su analize prolaznih podataka za pravljenje upotrebno-definisanih lanaca, zavisnih analiza, alijas analiza, pokaznih analiza, iskejp analiza itd.`
- error: `ird` → should be `itd` (type: substitution)
- LLM predicted: `Tipične analize su analize prolaznih podataka za pravljenje upotrebno-definisanih lanaca, zavisnih analiza, alijas analiza, pokaznih analiza, iskap analiza itd.`

**Example 9**

- corrupted: `Potom se tesko razboleo i celo telo mu je bilo pokriveno ranama od temena do tabana.`
- original: `Potom se teško razboleo i celo telo mu je bilo pokriveno ranama od temena do tabana.`
- error: `tesko` → should be `teško` (type: diacritic)
- LLM predicted: `Potom se teško razboleo i celo telo mu je bilo pokriveno ranama od temena do tabana.`

**Example 10**

- corrupted: `Alat je prvobitno nosio ime Worldcraft, dok ga je Ben Moris nezavisno razvijao, pre nego sto ga je Valv otkupio.`
- original: `Alat je prvobitno nosio ime Worldcraft, dok ga je Ben Moris nezavisno razvijao, pre nego što ga je Valv otkupio.`
- error: `sto` → should be `što` (type: diacritic)
- LLM predicted: `Alat je prvobitno nosio ime Worldcraft, dok ga je Ben Moris nezavisno razvijao, pre nego što ga je Valv otkupio.`


## (b) Cases the n-gram model fixed but Approach 1 (Norvig) got wrong

**Example 1**

- corrupted: `Naziv Stari Vlab se odnosi na Latine, a ne na Vlahe (narod).`
- original: `Naziv Stari Vlah se odnosi na Latine, a ne na Vlahe (narod).`
- error: `Vlab` → should be `Vlah` (type: substitution)
- Norvig predicted: `Slab`
- N-gram predicted: `Vlah`

**Example 2**

- corrupted: `Grnčarija ovog porekla ima nesumnjivu estetsku vrednost, a posebno sudovi u oblikg životinjskih figura.`
- original: `Grnčarija ovog porekla ima nesumnjivu estetsku vrednost, a posebno sudovi u obliku životinjskih figura.`
- error: `oblikg` → should be `obliku` (type: substitution)
- Norvig predicted: `oblik`
- N-gram predicted: `obliku`

**Example 3**

- corrupted: `Ustanova je počela sa radm 2. oktobra 1969.`
- original: `Ustanova je počela sa radom 2. oktobra 1969.`
- error: `radm` → should be `radom` (type: deletion)
- Norvig predicted: `radi`
- N-gram predicted: `radom`

**Example 4**

- corrupted: `Uglavnom je sivo-plave boje, dok su krila smeđa sa crim mrljama.`
- original: `Uglavnom je sivo-plave boje, dok su krila smeđa sa crnim mrljama.`
- error: `crim` → should be `crnim` (type: deletion)
- Norvig predicted: `rim`
- N-gram predicted: `crnim`

**Example 5**

- corrupted: `Precesija je tendencija žiroskopa da se zaokrene pod praim uglom prema ulaznoj sili poremećaja.`
- original: `Precesija je tendencija žiroskopa da se zaokrene pod pravim uglom prema ulaznoj sili poremećaja.`
- error: `praim` → should be `pravim` (type: deletion)
- Norvig predicted: `prvim`
- N-gram predicted: `pravim`

**Example 6**

- corrupted: `Međutim, ukradeni instrumenti se često budu pronađeni, nakon što su vođeni kao nestali duig niz godina.`
- original: `Međutim, ukradeni instrumenti se često budu pronađeni, nakon što su vođeni kao nestali dugi niz godina.`
- error: `duig` → should be `dugi` (type: transposition)
- Norvig predicted: `dug`
- N-gram predicted: `dugi`

**Example 7**

- corrupted: `Nauka koja proučava vrednosti naziav se aksiologija.`
- original: `Nauka koja proučava vrednosti naziva se aksiologija.`
- error: `naziav` → should be `naziva` (type: transposition)
- Norvig predicted: `naziv`
- N-gram predicted: `naziva`

**Example 8**

- corrupted: `Menjanje nalepnica u Braziu na Svetskom prvenstvu 2018.`
- original: `Menjanje nalepnica u Brazilu na Svetskom prvenstvu 2018.`
- error: `Braziu` → should be `Brazilu` (type: deletion)
- Norvig predicted: `Brazil`
- N-gram predicted: `Brazilu`

**Example 9**

- corrupted: `Teo postaje pravi frajet.`
- original: `Teo postaje pravi frajer.`
- error: `frajet` → should be `frajer` (type: substitution)
- Norvig predicted: `frajt`
- N-gram predicted: `frajer`

**Example 10**

- corrupted: `Da bi se prilagodila ovim promenama, birokratija je bila proširena i raznovrsna, da bi imala mnogo veću ulogu u upravj carstva.`
- original: `Da bi se prilagodila ovim promenama, birokratija je bila proširena i raznovrsna, da bi imala mnogo veću ulogu u upravi carstva.`
- error: `upravj` → should be `upravi` (type: substitution)
- Norvig predicted: `upravo`
- N-gram predicted: `upravi`


## (c) Cases everything failed on

**Example 1**

- corrupted: `Bio je i potpredsednik tv.`
- original: `Bio je i potpredsednik tzv.`
- error: `tv` → should be `tzv` (type: deletion)
- predictions: norvig=`(unchanged)`, ngram=`(unchanged)`, llm=`(unchanged)`

**Example 2**

- corrupted: `Osnovni zahtevi u Nemačkoj ticali su se garantija prava građama, donošenja ustava i ujedinjenja Nemačke na federalnim osnovama.`
- original: `Osnovni zahtevi u Nemačkoj ticali su se garantija prava građana, donošenja ustava i ujedinjenja Nemačke na federalnim osnovama.`
- error: `građama` → should be `građana` (type: substitution)
- predictions: norvig=`(unchanged)`, ngram=`(unchanged)`, llm=`građanima`

**Example 3**

- corrupted: `Ova dvojica slikara, potpuno različitih umetničkih shvatanja, prvi put su se sreli kao saradnici na izradi ikona za duvošku crkvu.`
- original: `Ova dvojica slikara, potpuno različitih umetničkih shvatanja, prvi put su se sreli kao saradnici na izradi ikona za divošku crkvu.`
- error: `duvošku` → should be `divošku` (type: substitution)
- predictions: norvig=`devojku`, ngram=`duboku`, llm=``

**Example 4**

- corrupted: `Takmičarke čija su iena podebljana učestvuju na EP 1996.`
- original: `Takmičarke čija su imena podebljana učestvuju na EP 1996.`
- error: `iena` → should be `imena` (type: deletion)
- predictions: norvig=`(unchanged)`, ngram=`(unchanged)`, llm=`jedna`

**Example 5**

- corrupted: `Godine 2001, objavljen je album Pišanje uz vetar na kome su se našle pesme kao što su Sbin je lud, Ljubav ovde više ne stanuje, Daj mi lovu itd.`
- original: `Godine 2001, objavljen je album Pišanje uz vetar na kome su se našle pesme kao što su Srbin je lud, Ljubav ovde više ne stanuje, Daj mi lovu itd.`
- error: `Sbin` → should be `Srbin` (type: deletion)
- predictions: norvig=`Sin`, ngram=`Sin`, llm=`Šbin`

**Example 6**

- corrupted: `Radi honorarno u butiku obuće Gles sbiper.`
- original: `Radi honorarno u butiku obuće Gles sliper.`
- error: `sbiper` → should be `sliper` (type: substitution)
- predictions: norvig=`skiper`, ngram=`biper`, llm=`Šiper`

**Example 7**

- corrupted: `Bend je počeo sa svirkom i onda se iz pozadine, kao deus ex machna cuo bubanj.`
- original: `Bend je počeo sa svirkom i onda se iz pozadine, kao deus ex machna čuo bubanj.`
- error: `cuo` → should be `čuo` (type: diacritic)
- predictions: norvig=`(unchanged)`, ngram=`(unchanged)`, llm=``

**Example 8**

- corrupted: `Takođe se i pominje „pop sin Cujica”, a takođe i „`
- original: `Takođe se i pominje „pop sin Vujica”, a takođe i „`
- error: `Cujica` → should be `Vujica` (type: substitution)
- predictions: norvig=`Bujica`, ngram=`Šujica`, llm=`(unchanged)`

**Example 9**

- corrupted: `Potenciometar radi na principu uravnoteženja nepoznatog napona nasuprot poznatom naponu u osnom spoju.`
- original: `Potenciometar radi na principu uravnoteženja nepoznatog napona nasuprot poznatom naponu u mosnom spoju.`
- error: `osnom` → should be `mosnom` (type: deletion)
- predictions: norvig=`(unchanged)`, ngram=`(unchanged)`, llm=`(unchanged)`

**Example 10**

- corrupted: `U slučaju da selekcija Rusije kao domaćin narednog turnira zauzme jednu od dve poslednje pozicije umesto njih u nizi rang takmičenja ispala bi 14. plasirana selekcija.`
- original: `U slučaju da selekcija Rusije kao domaćin narednog turnira zauzme jednu od dve poslednje pozicije umesto njih u niži rang takmičenja ispala bi 14. plasirana selekcija.`
- error: `nizi` → should be `niži` (type: diacritic)
- predictions: norvig=`(unchanged)`, ngram=`(unchanged)`, llm=`nizu`


## (d) LLM overcorrections (changed an originally-correct word)

**Example 1**

- corrupted: `Naime, kao metamorfmafus, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak lii čarolije.`
- original: `Naime, kao metamorfmafus, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak ili čarolije.`
- LLM changed correct word: `metamorfmafus` → `metamorfmafs`
- LLM predicted: `Naime, kao metamorfmafs, ona je mogla da promeni svoj fizički izgled po volji, bez potrebe da koristi napitak ili čarolije.`

**Example 2**

- corrupted: `Osnovni zahtevi u Nemačkoj ticali su se garantija prava građama, donošenja ustava i ujedinjenja Nemačke na federalnim osnovama.`
- original: `Osnovni zahtevi u Nemačkoj ticali su se garantija prava građana, donošenja ustava i ujedinjenja Nemačke na federalnim osnovama.`
- LLM changed correct word: `garantija` → `garancija`
- LLM predicted: `Osnovni zahtevi u Nemačkoj ticali su se garancija prava građanima, donošenja ustava i ujedinjenja Nemačke na federalnim osnovama.`

**Example 3**

- corrupted: `Sada većina brodova ima unutrašnji motor s unutrašnjim sagorijevanjem koristeći blqago refinirani tip petroleja zvani bunker gorivo.`
- original: `Sada većina brodova ima unutrašnji motor s unutrašnjim sagorijevanjem koristeći blago refinirani tip petroleja zvani bunker gorivo.`
- LLM changed correct word: `koristeći` → `koji koristi blago`
- LLM predicted: `Sada većina brodova ima unutrašnji motor s unutrašnjim sagorijevanjem koji koristi blago refinirani tip petroleja zvan bunker gorivo.`

**Example 4**

- corrupted: `Njihovo stavljanje u promet uskalđeno je sa Zakonom o biocidnim proizvodima, čije sprovođenje kontroliušu Agencije za hemikalije zemalja koje su taj zakon donele.`
- original: `Njihovo stavljanje u promet uskalđeno je sa Zakonom o biocidnim proizvodima, čije sprovođenje kontrolišu Agencije za hemikalije zemalja koje su taj zakon donele.`
- LLM changed correct word: `uskalđeno` → `usklađeno`
- LLM predicted: `Njihovo stavljanje u promet usklađeno je sa Zakonom o biocidnim proizvodima, čije sprovođenje kontrolišu Agencije za hemikalije zemalja koje su taj zakon donele.`

**Example 5**

- corrupted: `Molekul glukoze zasladjuje rastvor, a dijelovi glukoznih lanaca daju viskoznu konzistenciju.`
- original: `Molekul glukoze zaslađuje rastvor, a dijelovi glukoznih lanaca daju viskoznu konzistenciju.`
- LLM changed correct word: `dijelovi` → `delovi`
- LLM predicted: `Molekul glukoze zaslađuje rastvor, a delovi glukoznih lanaca daju viskoznu konzistenciju.`

**Example 6**

- corrupted: `Poslednja faza bitke odigrala se u podnožiju brda, u dolini reke Morave, gde su se Srbi utvridli u dolini uz reku sa namerom da kontrolišu prelaze na reci.`
- original: `Poslednja faza bitke odigrala se u podnožiju brda, u dolini reke Morave, gde su se Srbi utvrdili u dolini uz reku sa namerom da kontrolišu prelaze na reci.`
- LLM changed correct word: `podnožiju` → `podnožju`
- LLM predicted: `Poslednja faza bitke odigrala se u podnožju brda, u dolini reke Morave, gde su se Srbi utvrdili u dolini uz reku sa namerom da kontrolišu prelaze na reci.`

**Example 7**

- corrupted: `Razvijač Afdvanced Warfare-a je kompanija Sledgehamer gejms.`
- original: `Razvijač Advanced Warfare-a je kompanija Sledgehamer gejms.`
- LLM changed correct word: `Sledgehamer` → `Sledgehammer`
- LLM predicted: `Razvijač Advanced Warfare-a je kompanija Sledgehammer Games.`

**Example 8**

- corrupted: `Marina Nikola je medju prvima 1927.`
- original: `Marina Nikola je među prvima 1927.`
- LLM changed correct word: `prvima` → `prvimama`
- LLM predicted: `Marina Nikola je među prvimama 1927.`

**Example 9**

- corrupted: `Tipične analize su analize prolaznih podataka za pravljenje upotrebno-definisanih lanaca, zavisnih analiza, alijas analiza, pokaznih analiza, iskejp analiza ird.`
- original: `Tipične analize su analize prolaznih podataka za pravljenje upotrebno-definisanih lanaca, zavisnih analiza, alijas analiza, pokaznih analiza, iskejp analiza itd.`
- LLM changed correct word: `iskejp` → `iskap`
- LLM predicted: `Tipične analize su analize prolaznih podataka za pravljenje upotrebno-definisanih lanaca, zavisnih analiza, alijas analiza, pokaznih analiza, iskap analiza itd.`

**Example 10**

- corrupted: `U mestu je pravoslavna Vazneenska crkva, pri kojoj služe parosi Toma Bogdanović i Toma Lazić.`
- original: `U mestu je pravoslavna Vaznesenska crkva, pri kojoj služe parosi Toma Bogdanović i Toma Lazić.`
- LLM changed correct word: `parosi` → `parohi`
- LLM predicted: `U mestu je pravoslavna Vaznesenska crkva, pri kojoj služe parohi Toma Bogdanović i Toma Lazić.`



## Rezime

- **Pristup 1 (rečnik + Levenštajnovo rastojanje)** je najbrži (~95
  rečenica/sek) i najjednostavniji, ali postiže i najnižu tačnost ispravke
  (0.69). Ne koristi kontekst rečenice, pa ne može da otkrije grešku koja
  slučajno formira drugu, već poznatu reč (real-word greška, npr. `seno`
  umesto `selo`), i najviše se muči kod brisanja karaktera, gde je više
  poznatih reči podjednako blizu oštećenoj.

- **Pristup 2 (n-gram rerangiranje)** koristi identičnu detekciju i
  kandidate kao Pristup 1, samo bira među njima na osnovu bigramskog
  konteksta umesto sirove učestalosti. Ta jedna promena diže tačnost na
  0.73, dosledno na svih pet tipova grešaka, i gotovo nikad ne pogorša ono
  što je Pristup 1 već dobro pogodio (svega 14 od 106 diskordantnih
  slučajeva) - čist, jeftin dobitak, bez ikakvog kompromisa u brzini.

- **Pristup 3 (LLM zero-shot)** ima najveću ukupnu tačnost (0.78) i najveći
  detekcioni recall (0.94), jer ne zavisi od toga da li je reč "poznata"
  već radi nad celom rečenicom - zbog toga je i jedini koji hvata real-word
  greške nevidljive Pristupu 1 i 2. Ali "najbolji u proseku" ne znači i
  nepogrešiv: LLM je slabiji od oba lokalna pristupa baš kod umetanja i
  transpozicije, gde je tačna reč već očigledna na rastojanju uređivanja 1,
  sklon je povremenoj prekomernoj ispravci (menja i reči koje uopšte nisu
  bile greška), i oko 38 puta je sporiji od druga dva pristupa, jer svaka
  rečenica ide preko mrežnog poziva ka API-ju.

Nijedan pristup nije strogo dominantan: čak i u zajedničkom podskupu
rečenica, Rečnik i Ngram tačno pogađaju desetine grešaka koje LLM promaši - što ukazuje da bi kombinacija pristupa, na primer LLM uz
Ngram kao proveru za tipove grešaka gde LLM slabije prolazi, verovatno
nadmašila svaki pristup pojedinačno.